In [ ]:
# Import
import os
import pandas as pd
import numpy as np

from pathlib import Path

from CellPacking.tissuegeneration import sheet_init, symetric_circular
from CellPacking.dynamics import (Compression, 
                                  AnisotropicLineTension, 
                                  ShearPlanarGeometry, 
                                  PlaneBarrierElasticity)

from tyssue import Sheet
from tyssue import PlanarGeometry
from tyssue.solvers import QSSolver
from tyssue.solvers.viscous import EulerSolver
from tyssue.behaviors.event_manager import EventManager
from tyssue.behaviors.sheet.basic_events import reconnect, reconnect_3D
from tyssue.dynamics import model_factory, effectors
from tyssue.core.history import HistoryHdf5 


import matplotlib.pyplot as plt
from tyssue.draw import sheet_view
from CellPacking.plot import superimpose_sheet_view
from CellPacking.plot import sheet_view as ply_sheet_view

from tyssue.generation import extrude 
from tyssue import Monolayer
from CellPacking.dynamics import ShearMonolayerGeometry
from tyssue.io.hdf5 import save_datasets

from tyssue.io.hdf5 import load_datasets
from tyssue.io.meshes import save_triangular_mesh

In [ ]:
SIM_DIR = Path('/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/')

In [ ]:
folder = ["20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_2_5",
         "20250317_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3",
         "20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3_5"]
nb_repeat = [8,
             8,
            11]
ratios = np.arange(2, 4.2, 0.2)


In [ ]:
result = pd.DataFrame(columns = ["si_lateral", "repeat", "si_plane", "height_begin", "height_end"])

for i in range (3):
    for j in range(nb_repeat[i]):
        for r in ratios:
            sim_save_dir = SIM_DIR/folder[i]/str(j)/str(round(r,1))

            monolayer_d = load_datasets(os.path.join(sim_save_dir,'monolayer2.hf5'))
            monolayer = Monolayer("mono", monolayer_d)
            height_begin = (np.mean(monolayer.vert_df[monolayer.vert_df["segment"]=="apical"]["z"]) - 
                      np.mean(monolayer.vert_df[monolayer.vert_df["segment"]=="basal"]["z"]) )


            
            monolayer_d = load_datasets(os.path.join(sim_save_dir,'monolayer299.hf5'))
            monolayer = Monolayer("mono", monolayer_d)
            height_end = (np.mean(monolayer.vert_df[monolayer.vert_df["segment"]=="apical"]["z"]) - 
                      np.mean(monolayer.vert_df[monolayer.vert_df["segment"]=="basal"]["z"]) )

            if i == 0:
                sil = 2.5
            elif i == 1:
                sil = 3
            else :
                sil = 3.5
            result = pd.concat([result, pd.DataFrame({"si_lateral":sil, 
                                                     "repeat":j, 
                                                     "si_plane":round(r,1), 
                                                     "height_begin":height_begin,
                                                     "height_end":height_end},
                          index=[0])],
                          ignore_index=True)
result.to_csv(os.path.join(SIM_DIR, "height.csv"))

In [ ]:
result

In [ ]:
import random
get_colors = lambda n: ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(n)]
color_list = get_colors(36)

fig, ax = plt.subplots()
for r in [2.5, 3, 3.5]:
    ax.scatter(r,r, color=np.array(color_list)[int(r*10)])

In [ ]:

fig, ax = plt.subplots()
for i in [2.5, 3, 3.5]:
    map_ = ax.scatter(result[result["si_lateral"]==i]["si_plane"], 
                      result[result["si_lateral"]==i]["height_end"], 
                      color=np.array(color_list)[((result[result["si_lateral"]==i]["si_lateral"]*10)).astype(int).to_numpy()], 
                    label=str(i)
                      )
fig.legend()
fig.legends[0].set_title("lateral shape index")
ax.set_xlabel("plane shape index")
ax.set_ylabel("height")

fig.savefig(os.path.join(SIM_DIR, "height_end.png"))
fig.savefig(os.path.join(SIM_DIR, "height_end.eps"))

In [ ]:
fig, ax = plt.subplots()

map_ = ax.scatter(result.groupby(["si_lateral", "si_plane"]).mean().reset_index()["si_plane"], 
                  result.groupby(["si_lateral", "si_plane"]).mean()["height"], 
                 color=np.array(color_list)[((result.groupby(["si_lateral", "si_plane"]).mean().reset_index()["si_lateral"]*10)).astype(int).to_numpy()]
                 )
ax.legend()


In [ ]:

fig, ax = plt.subplots()
for i in [2.5, 3, 3.5]:
    map_ = ax.scatter(result[result["si_lateral"]==i]["si_plane"], 
                      result[result["si_lateral"]==i]["height_begin"], 
                      color=np.array(color_list)[((result[result["si_lateral"]==i]["si_lateral"]*10)).astype(int).to_numpy()], 
                    label=str(i)
                      )
fig.legend()
fig.legends[0].set_title("lateral shape index")
ax.set_xlabel("plane shape index")
ax.set_ylabel("height")

fig.savefig(os.path.join(SIM_DIR, "height_begin.png"))
fig.savefig(os.path.join(SIM_DIR, "height_begin.eps"))

In [ ]:

fig, ax = plt.subplots()
for i in [2.5, 3, 3.5]:
    map_ = ax.scatter(result[result["si_lateral"]==i]["si_plane"], 
                      result[result["si_lateral"]==i]["height_begin"]-result[result["si_lateral"]==i]["height_end"], 
                      color=np.array(color_list)[((result[result["si_lateral"]==i]["si_lateral"]*10)).astype(int).to_numpy()], 
                    label=str(i)
                      )
fig.legend()
fig.legends[0].set_title("lateral shape index")
ax.set_xlabel("plane shape index")
ax.set_ylabel("height")

fig.savefig(os.path.join(SIM_DIR, "height_diff_begin-end.png"))
fig.savefig(os.path.join(SIM_DIR, "height_diff_begin-end.eps"))

In [ ]:
paths = ["/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_2_5/",
        "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3/",
        "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3_5/", 
        "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_4/"]

shi = [2.5, 3, 3.5, 4]

res_dataframe = []

i=0
for sdir in paths : 
    result = pd.read_csv(os.path.join(sdir, 'result_count.csv'))
    result_sum = pd.read_csv(os.path.join(sdir, "result_ti.csv"))
    result_final = result.copy(deep=True)
    result_final['P'] = ((result_sum['Lambda_y']-result_sum['Lambda_x'])/result_final['tot_cell']*100).to_numpy()
    result_final.drop(['Unnamed: 0.1', 'Unnamed: 0',"pourcentage2", "pourc_change"], axis=1, inplace=True	)
    result_final["shape_index_lateral"] = shi[i]

    if i ==0 : 
        res_dataframe = result_final.copy(deep=True)
    else:
        res_dataframe = pd.concat([res_dataframe, result_final])
    i+=1
res_dataframe.reset_index(inplace=True, drop=True)
res_dataframe.to_csv("/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/result_all.csv")